In [1]:
# @title Step 0: Setup, Installation, and api key management

# Add your Google API Key
GOOGLE_MAPS_API_KEY="<INSERT YOUR GOOGLE MAPS API KEY HERE>"
PROJECT_ID = "<INSERT YOUR PROJECT_ID HERE>"

# Install ADK and LiteLLM
!pip install google-adk -q
!pip install litellm -q

print("Installation complete.")

Installation complete.


In [2]:
# @title Step 1: Imports, Settings and Constants
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm # For multi-model support
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts
from google.adk.tools import google_maps_grounding

# More supported models can be referenced here: https://ai.google.dev/gemini-api/docs/models#model-variations
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

print("\nEnvironment configured.")



Environment configured.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [3]:
# @title Step 2: get_current_weather(lat, lon)
import requests

def get_current_weather(lat, lon):
    """
    Retrieves a detailed current weather forecast for a specific geographic
    location using the National Weather Service (NWS) API.

    This function implements the required two-stage asynchronous workflow of the
    NWS API (api.weather.gov). Because the NWS organizes data by a "Grid
    Coordinate System" rather than direct GPS coordinates, the function must
    perform a discovery step before data retrieval.

    Internal Logic and Execution Flow:
        1. Metadata Resolution: The function targets the '/points' endpoint using
           the provided {lat},{lon}. This request does not return weather data;
           instead, it returns a metadata object containing the specific
           Forecast Office ID, Grid X, and Grid Y coordinates.
        2. Dynamic URL Extraction: From the metadata response, the function
           extracts the 'forecast' property, which is a unique URL pointing to
            the exact grid-square's time-series data.
        3. Forecast Fetching: The function executes a secondary GET request to
           the dynamically discovered forecast URL.
        4. Period Parsing: The NWS returns a list of 'periods' (e.g., Today,
           Tonight, Tomorrow). This function isolates the 0th index, representing
           the most immediate meteorological state.

    Args:
        lat (float or str): Latitude of the target location. Value must be within
            the geographic boundaries of the United States and its territories.
        lon (float or str): Longitude of the target location. Value must be within
            the geographic boundaries of the United States and its territories.

    Returns:
        str: A human-readable summary combining the name of the current time
            period and the 'detailedForecast' string (e.g., "Tonight: Mostly clear,
            with a low around 55. South wind 5 to 7 mph.").

    Technical Constraints & Requirements:
        - Protocol: HTTPS GET.
        - Headers: The NWS API strictly mandates a 'User-Agent' header. Failure
          to provide this will result in a 403 Forbidden error.
        - Data Format: JSON (Application/geo+json).
        - Scope: Non-US coordinates will return a 404 or a "Data Out of Bounds"
          error message within the JSON payload.
        - External Library: Requires the 'requests' package.
    """
    # NWS API requires a User-Agent header (use your app name or email)
    headers = {'User-Agent': '(weather_agent_tools, daniel.w.carey@saic.com)'}

    # Step 1: Get the metadata/gridpoint for the coordinates
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    res = requests.get(points_url, headers=headers).json()

    # Step 2: Get the forecast URL from the metadata
    forecast_url = res['properties']['forecast']
    forecast = requests.get(forecast_url, headers=headers).json()

    # Return the current period's status (e.g., "Sunny", "Mostly Cloudy")
    current_period = forecast['properties']['periods'][0]
    return f"{current_period['name']}: {current_period['detailedForecast']}"

# Example Usage:
# print(get_current_weather(38.8894, -77.0352))


In [4]:
# @title Step 3: get_location_lat_long

import requests

def get_location_lat_long(city: str, state: str) -> dict[str, float | None]:
    """
    Use the Google Maps Geocoding API to convert a city and state into a location object.

    Args:
        city: The city name, such as "Knoxville".
        state: The state abbreviation or name, such as "TN" or "Tennessee".

    Returns:
        A dictionary containing the latitude and longitude.

        Example:
        {
            "Latitude": 35.9606384,
            "Longitude": -83.9207392
        }

        If the location cannot be found or the API request fails, both values will be None:

        {
            "Latitude": None,
            "Longitude": None
        }
    """

    address = f"{city}, {state}"

    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": GOOGLE_MAPS_API_KEY
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print(f"Geocoding API error: {response.status_code}")
        return {
            "Latitude": None,
            "Longitude": None
        }

    data = response.json()

    if data.get("status") != "OK" or not data.get("results"):
        print(f"Geocoding API returned no results: {data.get('status')}")
        return {
            "Latitude": None,
            "Longitude": None
        }

    location = data["results"][0]["geometry"]["location"]

    return {
        "Latitude": location["lat"],
        "Longitude": location["lng"]
    }


# test
result = get_location_lat_long("knoxville", "tn")
print(result)

{'Latitude': 35.965266, 'Longitude': -83.923304}


In [5]:
# @title Step 4: Define the Weather Agent
AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

instruction = """
              You are a helpful weather assistant.
              When the user asks for the weather in a specific city,
              use the 'get_location_lat_long' function to get the latitude and longitude coordinates of the city,
              then pass those cooridinates to the 'get_current_weather' tool to get the weather information.
              If the tool returns an error, inform the user politely.
              If the tool is successful, return the weather report
              """

weather_agent = Agent(
    name="weather_agent_v1",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Provides weather information for specific cities.",
    instruction=instruction,
    tools=[get_location_lat_long, get_current_weather], # Pass the function directly
)

print(f"Agent '{weather_agent.name}' created using model '{AGENT_MODEL}'.")




Agent 'weather_agent_v1' created using model 'gemini-2.5-flash'.


In [6]:
# @title Step 5: Use vertexai with weather_agent

from vertexai.preview import reasoning_engines

app = reasoning_engines.AdkApp(
    agent=weather_agent,
    enable_tracing=False
)

In [7]:
# @title Step 6: Create session
user_id = "test-user-id"
session = app.create_session(user_id=user_id)

print(session["id"])
print(session)

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.


b13ef15f-3e51-4b89-9cad-1a87a89b4e6e
{'id': 'b13ef15f-3e51-4b89-9cad-1a87a89b4e6e', 'app_name': 'default-app-name', 'user_id': 'test-user-id', 'state': {}, 'events': [], 'last_update_time': 1783535882.7465174}


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:872: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [8]:
# @title Step 7: callTheWeatherAgent function
from IPython.display import Markdown, display

def callTheWeatherAgent(prompt:str):
  response = "sorry. i have no response"
  for event in app.stream_query(
      user_id=user_id,
      session_id=session["id"],
      message=prompt,
  ):
    lastevent = event
    content = lastevent.get("content", {})
    parts = content.get("parts", [])

    if parts and "text" in parts[0]:
      response = parts[0]["text"]

  return response



In [9]:
# @title Test the Weather Agent

test_prompts = [
  "What is the weather in Miami, FL?",
  "What is the weather in London England?",  # Makes sure works outside U.S.
  "What is the weather in Aspen Colorado?",
  "What is the weather near the mars rover?", # Ask about out of expected range
  "Who voiced Donald Duck?", # Ask outside of expected range
]

# Test prompt+responses
for prompt in test_prompts:
  agent_response = callTheWeatherAgent(prompt)
  print(f"user: {prompt}")
  print(f"agent: {agent_response}")
  print()

# Expected results:
# =================
#
# user: What is the weather in Miami, FL?
# agent: The weather in Miami, FL is: Today: Sunny, with a high near 90. Heat index values as high as 105. Southeast wind 8 to 13 mph.
#
# user: What is the weather in London England?
# agent: I can only provide weather information for locations within the United States.
#
# user: What is the weather in Aspen Colorado?
# agent: The weather in Aspen, Colorado is: This Afternoon: A slight chance of showers and thunderstorms. Mostly sunny, with a high near 87. North northwest wind around 5 mph. Chance of precipitation is 40%.
#
# user: What is the weather near the mars rover?
# agent: I cannot provide weather information for locations outside of Earth.

# user: Who voiced Donald Duck?
# agent: I am sorry, I can only provide information about the weather.

user: What is the weather in Miami, FL?
agent: Today: Sunny, with a high near 90. Heat index values as high as 105. Southeast wind 8 to 13 mph.

user: What is the weather in London England?
agent: I am sorry, but I can only provide weather information for cities within the United States.

user: What is the weather in Aspen Colorado?
agent: This Afternoon: A slight chance of showers and thunderstorms. Mostly sunny, with a high near 87. North northwest wind around 5 mph. Chance of precipitation is 40%.

user: What is the weather near the mars rover?
agent: I am sorry, but I can only provide weather information for locations on Earth, specifically within the United States. I cannot provide weather information for Mars.

user: Who voiced Donald Duck?
agent: I am sorry, but I am a weather assistant and can only provide information about the weather. I cannot answer questions about voice actors.

